# The Ledger on a real model: SmolLM2-135M-Instruct on IFEval (Kaggle)

Freezes SmolLM2-135M-Instruct and compares, on the official IFEval (541 prompts): `base` (the frozen model),
`lora` (the standard adaptation, with the same 3.96M trainable parameters and data as the Ledger), `ledger` (the
proposed architecture) and `ledger_joint` (the same slots read inside the model's own softmax), with each request
alone and followed by 6,000 tokens of unrelated text.

**Settings** (right panel): *Accelerator: GPU T4 x2*, *Internet: on*.
1. `STUDY = 'pilot'`, *Run All* (about 30 to 60 minutes); download the zip linked at the end and send it.
2. `STUDY = 'kaggle'`, *Save Version → Save & Run All (Commit)*; the browser can be closed. The zip is in the
   version's *Output* tab.
3. The house-pattern study (two example answers show a pattern the model must keep unasked, for 3 to 12 items):
   set `TASK = 'patterns'` and repeat steps 1 and 2.
4. If the time budget ends it first: *Add Input → Your Work → this notebook*, then *Save & Run All* again.

In [ ]:
# The Ledger on SmolLM2-135M-Instruct, on Kaggle.
# Settings panel (right): Accelerator = GPU T4 x2, Internet = on (needs a phone-verified Kaggle account).
#
# TASK = 'ifeval' (the IFEval study) or 'patterns' (the house-pattern study); both follow the same two steps.
# 1. STUDY = 'pilot', then Run All (about 30-60 minutes). Download the zip it links at the end and send it.
# 2. STUDY = 'kaggle', then Save Version > Save & Run All (Commit). The browser can be closed.
#    When it finishes: the notebook's Output tab > the zip. Send it.
# If the time budget ends a run before the study is done: Add Input > Your Work > this notebook (its latest
# version), then Save & Run All again. It copies the finished steps from that input and continues.

STUDY = 'pilot'        # 'pilot', then 'kaggle' ('full' = the paper's whole protocol, several runs)
TASK = 'ifeval'        # 'ifeval', or 'patterns': the house-pattern study (rules shown by two example answers, never stated)
TIME_BUDGET_H = 8.5    # stop cleanly, and pack what is done, before Kaggle's session limit

import datetime, json, os, pathlib, platform, shutil, signal, subprocess, sys, threading, time, urllib.request, zipfile

T0 = time.time()
KAGGLE_INPUT, WORKING = '/kaggle/input', '/kaggle/working'
REPO = '/tmp/ivi-137.github.io'
CODE = f'{REPO}/research/ledger-smollm'
WORK = f'{WORKING}/ledger-work'
SCRIPT, ANALYZE, PREFIX = {'ifeval': ('run_all.sh', 'analyze.py', ''), 'patterns': ('run_patterns.sh', 'pattern_analyze.py', 'patterns-')}[TASK]
OUT = f'{WORK}/{PREFIX}{STUDY}'


def sh(cmd, cwd=None):
    """Run a setup command, streaming its output; stop on failure."""
    p = subprocess.Popen(cmd, shell=True, cwd=cwd, text=True, bufsize=1, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    for line in p.stdout:
        print(line, end='', flush=True)
    if p.wait():
        raise RuntimeError(f'failed with exit code {p.returncode}: {cmd}')


# ── 1. GPU, internet, code, tests ──
def gpus():
    """(name, GB) per GPU, from nvidia-smi, so that this notebook's own process holds no GPU memory."""
    q = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader,nounits'], capture_output=True, text=True)
    return [(n.strip(), int(m) / 1024) for n, m in (line.rsplit(',', 1) for line in q.stdout.strip().splitlines())] if q.returncode == 0 else []


try:
    GPUS = gpus()
except FileNotFoundError:
    GPUS = []
if not GPUS:
    raise SystemExit('No GPU: Settings > Accelerator > GPU T4 x2, then run again.')
N_GPU, GPU_GB = len(GPUS), min(gb for _, gb in GPUS)
print(f'{N_GPU} GPU(s): {GPUS[0][0]}, {GPU_GB:.0f} GB each')
try:
    urllib.request.urlopen('https://github.com', timeout=15)
except Exception as e:
    raise SystemExit(f'No internet ({e}): Settings > Internet > on, then run again.')
if not os.path.isdir(REPO):
    sh(f'git clone --depth 1 https://github.com/ivi-137/ivi-137.github.io {REPO}')
else:
    sh(f'git -C {REPO} pull --ff-only')
sh(f'{sys.executable} -m pip install -q -r requirements.txt', cwd=CODE)
sh(f'{sys.executable} setup_ifeval.py', cwd=CODE)        # official IFEval checkers and prompts, pinned commit
sh(f'{sys.executable} -m pytest -q -x tests', cwd=CODE)   # the implementation's tests, on a tiny random model
COMMIT = subprocess.run(['git', '-C', REPO, 'rev-parse', 'HEAD'], capture_output=True, text=True).stdout.strip()

# ── 2. resume: copy finished steps from an earlier version of this notebook added as input ──
os.makedirs(WORK, exist_ok=True)
for prev in sorted(pathlib.Path(KAGGLE_INPUT).glob('**/ledger-work')) if os.path.isdir(KAGGLE_INPUT) else []:
    copied = 0
    for root, _, files in os.walk(prev):
        dst = pathlib.Path(WORK, os.path.relpath(root, prev))
        dst.mkdir(parents=True, exist_ok=True)
        for f in files:
            if not (dst / f).exists():
                shutil.copyfile(os.path.join(root, f), dst / f)  # contents only: inputs are read-only
                copied += 1
    print(f'resumed {copied} files from {prev}')

# ── 3. the study: on two GPUs, one builds the data while the other scores the frozen model, then each trains
#       and scores its share of the seeds; every step is skipped if already done ──
ENV = dict(os.environ, PYTHON=sys.executable, DEVICE='cuda', BATCH='64', OUT=OUT, PYTHONUNBUFFERED='1',
           TOKENIZERS_PARALLELISM='false')
if TASK == 'patterns':  # reuse the IFEval study's background passages when an earlier version made them
    found = [f for f in (pathlib.Path(WORK, q, 'data', 'distractors.jsonl') for q in ('kaggle', 'full', 'pilot')) if f.exists()]
    if found:
        ENV['DISTRACTORS'] = str(found[0])
EVAL_TOKENS = 65536 if GPU_GB < 20 else 131072 if GPU_GB < 45 else 262144
plan = subprocess.run(['bash', SCRIPT, STUDY], cwd=CODE, env=dict(ENV, PLAN='1'), capture_output=True, text=True, check=True)
SEEDS, VARIANTS, LENGTHS = (line.split() for line in plan.stdout.splitlines())
stop, failed, procs = threading.Event(), [], set()


def run(tag, gpu, steps, seeds=None):
    """The study's runner limited to some steps, on one GPU; on failure (usually memory) retry with smaller evaluation batches."""
    tokens = EVAL_TOKENS
    while not stop.is_set():
        env = dict(ENV, CUDA_VISIBLE_DEVICES=str(gpu), STEPS=steps, EVAL_TOKENS=str(tokens), **({'SEEDS': seeds} if seeds else {}))
        p = subprocess.Popen(['bash', SCRIPT, STUDY], cwd=CODE, env=env, text=True, bufsize=1,
                             stdout=subprocess.PIPE, stderr=subprocess.STDOUT, start_new_session=True)
        procs.add(p)
        for line in p.stdout:
            print(f'[{tag}] {line}', end='', flush=True)
        code = p.wait()
        procs.discard(p)
        if code == 0:
            return True
        if stop.is_set():
            return False
        if tokens <= 8192:
            failed.append(f'{tag}: {steps}')
            return False
        tokens //= 2
        print(f'[{tag}] failed (exit {code}); retrying with evaluation batches of at most {tokens} tokens', flush=True)
    return False


def ready(path):
    """Wait until the other GPU has written `path`; False if the study stopped or the data step failed."""
    while not os.path.exists(path):
        if stop.is_set() or any(f.endswith(': data') for f in failed):
            return False
        time.sleep(15)
    return True


if N_GPU == 1:
    lanes = [lambda: run('gpu0', 0, 'data train base eval')]
else:
    mine = [' '.join(SEEDS[i::2]) for i in range(2)]  # seeds 0, 2 on the first GPU; seed 1 on the second
    lanes = [
        lambda: run('gpu0', 0, 'data') and (not mine[0] or run('gpu0', 0, 'train eval', mine[0])),
        # the frozen model's scores need the background passages (IFEval) or the test prompts (patterns)
        lambda: (ready(f'{OUT}/data/' + ('distractors.jsonl' if TASK == 'ifeval' else 'test.jsonl')) and run('gpu1', 1, 'base'),
                 mine[1] and ready(f'{OUT}/data/train.jsonl') and run('gpu1', 1, 'train eval', mine[1])),
    ]
threads = [threading.Thread(target=f) for f in lanes]
for t in threads:
    t.start()
deadline = T0 + TIME_BUDGET_H * 3600
while any(t.is_alive() for t in threads):
    time.sleep(10)
    if time.time() > deadline and not stop.is_set():
        print(f'\nTime budget of {TIME_BUDGET_H} h reached: stopping and packing what is done.', flush=True)
        stop.set()
        for p in list(procs):
            try:
                os.killpg(p.pid, signal.SIGTERM)
            except ProcessLookupError:
                pass
for t in threads:
    t.join()

# ── 4. statistics, status, and a zip to send back ──
res = pathlib.Path(OUT, 'results')
if any(res.glob('*-L*.jsonl')):
    try:
        sh(f'{sys.executable} {ANALYZE} --results "{res}"', cwd=CODE)
    except RuntimeError as e:
        print(e)  # pack the raw results anyway
expected = [f'base-L{L}.jsonl' for L in LENGTHS] + [f'{v}-s{s}-L{L}.jsonl' for s in SEEDS for v in VARIANTS for L in LENGTHS]
missing = [f for f in expected if not (res / f).exists()]
status = {'task': TASK, 'study': STUDY, 'complete': not missing, 'missing_results': missing, 'failed_steps': failed,
          'stopped_by_time_budget': stop.is_set(), 'hours': round((time.time() - T0) / 3600, 2),
          'gpus': [n for n, _ in GPUS], 'gpu_gb': round(GPU_GB, 1), 'eval_tokens': EVAL_TOKENS, 'commit': COMMIT,
          'python': platform.python_version(), 'finished': datetime.datetime.now().isoformat()}
versions = subprocess.run([sys.executable, '-c', 'import torch, transformers; print(torch.__version__, torch.version.cuda, transformers.__version__)'],
                          capture_output=True, text=True).stdout.split()
status.update(zip(('torch', 'cuda', 'transformers'), versions))
pathlib.Path(OUT, 'status.json').write_text(json.dumps(status, indent=2), encoding='utf-8')
zpath = pathlib.Path(WORKING, f'ledger-results-{PREFIX}{STUDY}-{datetime.datetime.now():%Y%m%d-%H%M}.zip')
with zipfile.ZipFile(zpath, 'w', zipfile.ZIP_DEFLATED) as z:
    for f in sorted(pathlib.Path(WORK).rglob('*')):
        # results (every response and verdict), summaries, logs, data statistics and status; not the weights
        if f.is_file() and (f.parent.name == 'results' or f.name in ('run.json', 'log.jsonl', 'stats.json', 'status.json')):
            z.write(f, str(f.relative_to(WORK)))
print(json.dumps({k: status[k] for k in ('task', 'study', 'complete', 'missing_results', 'failed_steps', 'hours')}, indent=1))
print(f'\n{zpath} ({zpath.stat().st_size / 2**20:.1f} MB): send this file.')
md = res / 'summary.md'
if md.exists():
    print('\n' + md.read_text(encoding='utf-8'))
try:
    from IPython.display import FileLink, display
    display(FileLink(os.path.relpath(zpath, WORKING)))  # a download link in interactive sessions
except Exception:
    pass
